# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a FAIR data package using the `mlcroissant` library, referencing all data entities by their `@id`.

### Dataset Source
The dataset is defined via a Croissant schema, accessible at:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and access its properties using the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Convert the metadata to a python dict for exploration
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")

## 2. Data Overview
Display available record sets, their `@id`s, fields, and columns. All Croissant entities are referenced using their `@id`.

See [mlcroissant docs](https://mlcommons.github.io/croissant/api/mlcroissant.Dataset.html) for more on introspection.

In [ ]:
# Explore available record sets in the dataset
record_set_metadatas = list(dataset.record_sets())
print("Record Sets (@id):")
record_set_ids = []
for rs in record_set_metadatas:
    print(f"  - {rs['@id']}: {rs.get('name', '')}")
    record_set_ids.append(rs['@id'])
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        for fld in fields:
            fld_id = fld['@id'] if isinstance(fld, dict) else fld
            print(f"      Field: {fld_id}")
    if 'column' in rs:
        columns = rs['column']
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            col_id = col['@id'] if isinstance(col, dict) else col
            print(f"      Column: {col_id}")
if not record_set_ids:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found in the previous cell.

If more than one record set is present, all will be loaded into a dictionary of DataFrames. We'll extract columns using their `@id`.

In [ ]:
# Load records from each record set into a Pandas DataFrame
dataframes = {}

if not record_set_ids:
    print("No record sets found; cannot extract records.")
else:
    for rs_id in record_set_ids:
        print(f"Loading records for RecordSet @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns in {rs_id}:\n", df.columns.tolist())
        print(df.head(2), "\n")
    # For demonstration, choose the first record set for further analysis
    main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter, normalize, and group by relevant fields referenced by their `@id`.

* For illustration, we select a numeric field (e.g., `GAD-7` total score) and a grouping field such as `village` if present, referencing columns by their Croissant `@id`. Adjust the field `@id` if your schema uses different identifiers.

In [ ]:
# Example field @id for numeric analysis (adjust as per your schema, e.g., 'gad7_total_score')
# You should check column names in main_record_set_df.columns or metadata

# We use the main_record_set_id from the previous cell
if record_set_ids:
    df = dataframes[main_record_set_id]
    # Guess a likely numeric field @id from common survey fields. Replace with actual @id.
    possible_numeric_ids = [
        'gad7_total_score', # Example: GAD-7
        'phq9_total_score', # Example: PHQ-9
        'psq_score',        # Example: PSQ
        'age',              # Demographics
    ]
    numeric_field_id = None
    for col in df.columns:
        if col in possible_numeric_ids:
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("Could not detect standard numeric field. Available columns:", df.columns.tolist())
        # Optionally, let user pick a field interactively
    else:
        print(f"Using numeric field @id: {numeric_field_id}")

        # Filter for values above threshold
        threshold = 10
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records ({numeric_field_id} > {threshold}):\n", filtered_df.head())

            # Normalize
            normalized_col = numeric_field_id + "_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, normalized_col]].head())

            # Try grouping by village or another categorical column
            possible_group_ids = [
                'village',
                'gender',
                'level_of_education',
            ]
            group_field_id = None
            for col in df.columns:
                if col in possible_group_ids:
                    group_field_id = col
                    break
            if group_field_id:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped mean {numeric_field_id} by '{group_field_id}':\n", grouped_df.head())
            else:
                print("No standard group field found (village, gender, level_of_education)")
        else:
            print(f"Field '{numeric_field_id}' does not appear to be numeric. Column dtype: {df[numeric_field_id].dtype}")

## 5. Visualization
Visualize the distribution of the selected numeric field, optionally split by a categorical variable, using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to discover and analyze a Croissant FAIR dataset using `mlcroissant` by referencing all structural elements by their `@id`. The sample analysis included filtering and normalizing key mental health survey measures, grouping by demographic field, and plotting distributions.

This workflow can be expanded and adapted based on your dataset's actual schema and research questions, always keeping references by the Croissant `@id` for interoperability and traceability.